# 04. ReAct Agent（模擬版）

學習 ReAct Agent 模式的核心概念（使用模擬回應，無需 API Key）。

---

## 🧠 什麼是 ReAct？

**ReAct = Reasoning + Acting** (推理 + 行動)

這是一個讓 AI Agent 像人類思考一樣的迴圈：

```
思考 (Thought) → 行動 (Action) → 觀察 (Observation) → 思考 → ...
```

### 實際例子

就像你解決問題時會：
1. **思考**：「我需要計算這個數學式」
2. **行動**：使用計算機
3. **觀察**：得到結果 14
4. **思考**：「接下來我要搜尋資料」
5. ...直到任務完成

## 📦 ReAct Agent 架構圖

```
┌─────────────────────────────────────────────────────────┐
│                    ReAct Agent 架構                      │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌─────────────┐                                       │
│   │   START     │                                       │
│   └──────┬──────┘                                       │
│          │                                              │
│          ▼                                              │
│   ┌─────────────┐     有工具要執行?       ┌─────────────┐ │
│   │    Agent    │ ─────────────────────▶│    Tools    │ │
│   │   決策節點    │         Yes           │ (執行工具)   │ │
│   └──────┬──────┘                       └──────┬──────┘ │
│          │ No                                  │        │
│          ▼                                     │        │
│   ┌─────────────┐                              │        │
│   │     END     │◀─────────────────────────────┘        │
│   └─────────────┘         回到 Agent 繼續思考             │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

**關鍵特性：**
- 🔄 **循環結構**：`agent → tools → agent` 形成推理迴圈
- 🚦 **條件邊**：根據 `tool_calls` 決定下一步
- 🛡️ **狀態追蹤**：用 `iteration` 防止無限迴圈

In [1]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

---

## 4.1 定義 Agent 狀態

狀態是 Agent 在執行過程中的「記憶」，包含：

| 欄位 | 用途 |
|------|------|
| `messages` | 對話歷史（使用 reducer 累加） |
| `current_step` | 目前步驟：thinking / acting / finished |
| `tool_calls` | 待執行的工具列表 |
| `tool_results` | 工具執行結果 |
| `iteration` | 迭代次數（防止無限迴圈） |

In [2]:
class AgentState(TypedDict):
    """Agent 狀態定義
    
    這個狀態結構追蹤 Agent 的完整執行過程
    """
    messages: Annotated[list, add_messages]  # 對話歷史
    current_step: str                        # 目前步驟
    tool_calls: list                         # 待執行工具
    tool_results: list                       # 工具結果
    iteration: int                           # 迭代計數器

print("✅ Agent 狀態定義完成")

✅ Agent 狀態定義完成


---

## 4.2 定義工具（模擬版）

工具是 Agent 可以使用的「能力」。這裡我們定義兩個模擬工具：

| 工具 | 功能 | 範例輸入 | 範例輸出 |
|------|------|---------|----------|
| `calculator` | 計算數學式 | `"2 + 3 * 4"` | `14` |
| `search` | 搜尋資料 | `"LangGraph"` | 模擬結果 |

In [3]:
def mock_calculator(expression: str) -> str:
    """模擬計算器工具
    
    Args:
        expression: 數學表達式，如 "2 + 3 * 4"
    Returns:
        計算結果字串
    """
    try:
        result = eval(expression)
        return f"計算結果: {result}"
    except:
        return "計算錯誤"

def mock_search(query: str) -> str:
    """模擬搜尋工具
    
    Args:
        query: 搜尋關鍵字
    Returns:
        搜尋結果字串
    """
    return f"搜尋 '{query}' 的結果: 這是模擬的搜尋結果。"

# 工具註冊表
TOOLS = {
    "calculator": mock_calculator,
    "search": mock_search
}

print(f"✅ 已註冊 {len(TOOLS)} 個工具: {list(TOOLS.keys())}")

✅ 已註冊 2 個工具: ['calculator', 'search']


---

## 4.3 Agent 節點（決策核心）

Agent 節點負責「思考」和「決策」：
- 分析當前狀態
- 決定要呼叫哪些工具
- 或判斷任務已完成

```
迭代 0 → 決定呼叫 calculator
迭代 1 → 決定呼叫 search  
迭代 2 → 決定結束
```

> 💡 **提示**：在實際應用中，這裡會換成 LLM 來進行智能決策

In [4]:
def mock_agent(state: AgentState) -> dict:
    """模擬 Agent 決策
    
    根據當前迭代次數，模擬不同的決策：
    - 迭代 0: 呼叫計算器
    - 迭代 1: 呼叫搜尋
    - 迭代 2+: 結束
    """
    iteration = state.get("iteration", 0)
    
    if iteration == 0:
        print("🤔 Agent 思考: 需要計算數學式...")
        return {
            "current_step": "thinking",
            "tool_calls": [{"tool": "calculator", "args": "2 + 3 * 4"}],
            "iteration": 1
        }
    elif iteration == 1:
        print("🤔 Agent 思考: 接下來搜尋相關資料...")
        return {
            "current_step": "acting",
            "tool_calls": [{"tool": "search", "args": "LangGraph tutorial"}],
            "iteration": 2
        }
    else:
        print("🤔 Agent 思考: 任務完成！")
        return {
            "current_step": "finished",
            "tool_calls": [],
            "iteration": iteration + 1
        }

print("✅ Agent 節點定義完成")

✅ Agent 節點定義完成


---

## 4.4 工具執行節點

這個節點負責：
1. 讀取 `tool_calls` 中的待執行工具
2. 依序執行每個工具
3. 收集結果到 `tool_results`
4. 清空 `tool_calls`（避免重複執行）

In [5]:
def execute_tools(state: AgentState) -> dict:
    """執行工具呼叫
    
    遍歷所有待執行的工具，執行並收集結果
    """
    results = []
    
    for call in state.get("tool_calls", []):
        tool_name = call["tool"]
        args = call["args"]
        
        if tool_name in TOOLS:
            result = TOOLS[tool_name](args)
            results.append({"tool": tool_name, "result": result})
            print(f"  🔧 執行 {tool_name}({args})")
            print(f"     → 結果: {result}")
        else:
            print(f"  ❌ 未知工具: {tool_name}")
    
    # 返回結果，並清空 tool_calls
    return {
        "tool_results": results,
        "tool_calls": []  # 清空，避免重複執行
    }

print("✅ 工具執行節點定義完成")

✅ 工具執行節點定義完成


---

## 4.5 建構 ReAct 圖

現在把所有元件組合成完整的圖：

```
START → agent ──(有工具)──→ tools ──→ agent ──(無工具)──→ END
                    └────────────────────────────┘
                           (循環直到完成)
```

In [6]:
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """條件路由：決定是否繼續執行工具
    
    Returns:
        "tools": 如果有待執行的工具
        "end": 如果沒有待執行的工具
    """
    if state.get("tool_calls"):
        return "tools"
    return "end"

# 建構圖
graph = StateGraph(AgentState)

# 添加節點
graph.add_node("agent", mock_agent)
graph.add_node("tools", execute_tools)

# 添加邊
graph.add_edge(START, "agent")                    # 入口
graph.add_conditional_edges("agent", should_continue, {
    "tools": "tools",                             # 有工具 → 執行工具
    "end": END                                    # 無工具 → 結束
})
graph.add_edge("tools", "agent")                  # 工具執行後 → 回到 agent

# 編譯
app = graph.compile()
print("✅ ReAct 圖建構完成")

✅ ReAct 圖建構完成


In [7]:
# 視覺化圖結構
print("📊 圖結構 (Mermaid 格式):")
print(app.get_graph().draw_mermaid())

📊 圖結構 (Mermaid 格式):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -. &nbsp;end&nbsp; .-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



![圖結構](<./images/agent.png>)

---

## 4.6 執行 Agent

現在讓我們執行 Agent，觀察完整的推理過程：

| 迭代 | Agent 決策 | 工具執行 | 結果 |
|------|-----------|---------|------|
| 0 | 呼叫 calculator | `2 + 3 * 4` | 14 |
| 1 | 呼叫 search | `LangGraph tutorial` | 模擬結果 |
| 2 | 無工具呼叫 | - | 結束 |

In [8]:
print("🚀 開始執行 ReAct Agent\n")
print("=" * 50)

result = app.invoke({
    "messages": [],
    "current_step": "start",
    "tool_calls": [],
    "tool_results": [],
    "iteration": 0
})

print("=" * 50)
print(f"\n📋 最終狀態:")
print(f"   • 狀態: {result['current_step']}")
print(f"   • 迭代次數: {result['iteration']}")
print(f"   • 最後工具結果: {result['tool_results']}")

🚀 開始執行 ReAct Agent

🤔 Agent 思考: 需要計算數學式...
  🔧 執行 calculator(2 + 3 * 4)
     → 結果: 計算結果: 14
🤔 Agent 思考: 接下來搜尋相關資料...
  🔧 執行 search(LangGraph tutorial)
     → 結果: 搜尋 'LangGraph tutorial' 的結果: 這是模擬的搜尋結果。
🤔 Agent 思考: 任務完成！

📋 最終狀態:
   • 狀態: finished
   • 迭代次數: 3
   • 最後工具結果: [{'tool': 'search', 'result': "搜尋 'LangGraph tutorial' 的結果: 這是模擬的搜尋結果。"}]


---

## 4.7 串流觀察執行過程

使用 `stream()` 可以逐步觀察 Agent 的執行過程：

In [9]:
print("🔄 串流執行過程:\n")

initial_state = {
    "messages": [],
    "current_step": "start",
    "tool_calls": [],
    "tool_results": [],
    "iteration": 0
}

for step_num, chunk in enumerate(app.stream(initial_state)):
    for node_name, values in chunk.items():
        print(f"步驟 {step_num + 1} | 節點: {node_name}")
        print(f"         | 狀態: {values.get('current_step', '-')}")
        print(f"         | 迭代: {values.get('iteration', '-')}")
        print()

🔄 串流執行過程:

🤔 Agent 思考: 需要計算數學式...
步驟 1 | 節點: agent
         | 狀態: thinking
         | 迭代: 1

  🔧 執行 calculator(2 + 3 * 4)
     → 結果: 計算結果: 14
步驟 2 | 節點: tools
         | 狀態: -
         | 迭代: -

🤔 Agent 思考: 接下來搜尋相關資料...
步驟 3 | 節點: agent
         | 狀態: acting
         | 迭代: 2

  🔧 執行 search(LangGraph tutorial)
     → 結果: 搜尋 'LangGraph tutorial' 的結果: 這是模擬的搜尋結果。
步驟 4 | 節點: tools
         | 狀態: -
         | 迭代: -

🤔 Agent 思考: 任務完成！
步驟 5 | 節點: agent
         | 狀態: finished
         | 迭代: 3



---

## 💡 重點學習

### ReAct 模式的關鍵特性

1. **循環結構**
   - `agent → tools → agent` 形成推理迴圈
   - 直到 Agent 判斷任務完成

2. **條件邊**
   - `should_continue` 函數決定走向
   - 有 `tool_calls` → 執行工具
   - 沒有 → 結束

3. **狀態追蹤**
   - `iteration` 計數器防止無限迴圈
   - `tool_results` 累積所有工具結果

4. **工具整合**
   - 用字典 `TOOLS` 管理所有工具
   - 統一的呼叫介面

### 實際應用

在真實場景中：
- `mock_agent` → 換成 LLM (如 GPT-4)
- `TOOLS` → 換成真實 API (搜尋、資料庫、計算...)
- 添加錯誤處理和重試機制

---

## 📝 練習題

1. **新增工具**：添加一個 `get_weather(city)` 工具
2. **修改決策邏輯**：讓 Agent 根據訊息內容決定使用哪個工具
3. **添加限制**：設定最大迭代次數為 5，超過就強制結束
4. **錯誤處理**：當工具執行失敗時，讓 Agent 嘗試其他方案

---

下一步：[05. Tool 整合](05_tools.ipynb)